# Versao 5 - Visao Geral dos Dados

## Proposito cientifico

Esta versao reorganiza o projeto em torno de uma pergunta comparativa central: sob um mesmo protocolo experimental, qual e o comportamento relativo de uma `LSTM` pura, de um modelo `hibrido residual` e da baseline de `persistencia` para previsao multi-step no dataset `3W`?

O presente notebook cumpre quatro objetivos academicos:

1. caracterizar formalmente a base `3W`;
2. explicitar a semantica dos atributos originais;
3. documentar a taxonomia dos eventos indesejaveis;
4. estabelecer uma base descritiva consistente para a fase de modelagem comparativa.

## Delimitacao metodologica

O estudo adota a base publica `3W Dataset 2.0.0`, publicada pela Petrobras, concebida para experimentacao em deteccao e classificacao de eventos indesejaveis em pocos offshore. Embora o dataset tenha sido originalmente estruturado para problemas de classificacao, neste projeto ele e reinterpretado como um problema de previsao supervisionada multivariada, em que o objetivo passa a ser antecipar a evolucao imediata de variaveis fisicas instrumentadas.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao5" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
from versao5.pipeline_v5 import (
    build_synthetic_attribute_catalog,
    load_attribute_catalog,
    load_event_catalog,
    summarize_dataset_inventory,
)

DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
inventory = summarize_dataset_inventory(DATASET_ROOT)
attribute_catalog = load_attribute_catalog(DATASET_ROOT)
event_catalog = load_event_catalog(DATASET_ROOT)
synthetic_catalog = build_synthetic_attribute_catalog()

inventory["overview"]

## Estrutura macro do dataset

A estrutura observada deve ser interpretada sob tres eixos de heterogeneidade:

- heterogeneidade de classe, pois o conjunto contem operacao normal e multiplos eventos indesejaveis;
- heterogeneidade de origem, uma vez que ha instancias reais de poco, instancias simuladas e instancias desenhadas;
- heterogeneidade temporal, porque o numero de observacoes por serie varia substancialmente.

Em um artigo cientifico, essa heterogeneidade deve ser explicitada porque ela afeta representatividade, complexidade do treinamento e interpretacao dos resultados.

In [ ]:
display(inventory["overview"])
display(inventory["class_distribution"])
display(inventory["source_distribution"])
display(inventory["well_distribution"].head(15))

if not inventory["length_distribution"].empty:
    display(
        inventory["length_distribution"]["n_linhas"].describe().to_frame("n_linhas")
    )

## Atributos originais

O catalogo abaixo reproduz, de forma rastreavel, a descricao oficial dos atributos disponibilizados no `dataset.ini` do `3W`. Para fins de redacao academica, recomenda-se distinguir entre:

- variaveis-alvo diretamente previstas pelo modelo;
- variaveis auxiliares analogicas;
- sinais discretos de estado;
- metadados e rotulos.

Essa separacao e importante porque nem todo atributo observado e automaticamente convertido em feature de entrada. A selecao final depende da etapa de pre-processamento e da variabilidade efetiva encontrada no conjunto de treino.

In [ ]:
display(attribute_catalog)

## Taxonomia dos eventos

O dataset organiza os arquivos em dez classes, das quais a classe `0` representa operacao normal e as demais representam regimes indesejaveis. Essa taxonomia e relevante mesmo em um problema de previsao, porque:

- ela permite avaliar o comportamento dos modelos sob dinamicas fisicas distintas;
- ela orienta o balanceamento amostral no treinamento;
- ela fundamenta analises estratificadas por classe no conjunto de teste.

In [ ]:
display(event_catalog)

cross_tab = (
    inventory["manifest"]
    .groupby(["class_label", "source_type"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
cross_tab

## Leitura academica dos dados

Do ponto de vista metodologico, tres observacoes merecem destaque:

1. O dataset e naturalmente desbalanceado. Consequentemente, comparacoes inguas de erro agregado podem mascarar desempenho insatisfatorio em subconjuntos raros.
2. A coexistencia de fontes reais, simuladas e desenhadas aumenta a diversidade do espaco amostral, mas tambem pode introduzir padroes com graus distintos de realismo operacional.
3. As variaveis possuem escalas fisicas muito diferentes, o que justifica a adocao posterior de escalonamento robusto, clipping quantilico e analise numerica das colunas continuas.

Em termos de redacao para artigo, este notebook fornece a secao descritiva da base, enquanto o notebook seguinte formaliza a engenharia de atributos e a construcao dos artefatos experimentais.

In [ ]:
summary_for_article = {
    "n_arquivos": int(inventory["overview"].loc[inventory["overview"]["estatistica"] == "arquivos", "valor"].iloc[0]),
    "n_classes": int(inventory["overview"].loc[inventory["overview"]["estatistica"] == "classes_distintas", "valor"].iloc[0]),
    "n_pocos": int(inventory["overview"].loc[inventory["overview"]["estatistica"] == "pocos_distintos", "valor"].iloc[0]),
    "n_fontes": int(inventory["overview"].loc[inventory["overview"]["estatistica"] == "tipos_de_origem", "valor"].iloc[0]),
}
summary_for_article